# LLaVA

``` text
图像
→ Vision Encoder
→ 视觉特征
→ Projector
→ LLM维度的视觉Token
→ 与文本Token拼接
→ Decoder-only LLM
→ 文本答案
```

In [ ]:
import torch
import torch.nn as nn


class SimpleLLaVA(nn.Module):
    def __init__(
        self,
        vision_encoder: nn.Module,
        language_model: nn.Module,
        vision_dim: int,
        language_dim: int,
    ):
        super().__init__()

        self.vision_encoder = vision_encoder
        self.language_model = language_model

        self.projector = nn.Sequential(
            nn.Linear(vision_dim, language_dim),
            nn.GELU(),
            nn.Linear(language_dim, language_dim),
        )

    def forward(
        self,
        images: torch.Tensor,
        input_ids: torch.Tensor,
        labels: torch.Tensor | None = None,
    ):
        # 1. 提取图像特征
        # [B, N, vision_dim]
        vision_features = self.vision_encoder(images)

        # 2. 图像经过视觉编码器 + 投影层，变成 LLM 能理解的 token
        # [B, N, language_dim]
        vision_embeddings = self.projector(
            vision_features
        )

        # 3. 文本经过语言模型的嵌入层，变成 LLM 能理解的 token
        # [B, T, language_dim]
        text_embeddings = (
            self.language_model
            .get_input_embeddings()(input_ids)
        )

        # 4. 拼接视觉与文本 token
        # [B, N+T, language_dim]
        multimodal_embeddings = torch.cat(
            [vision_embeddings, text_embeddings],
            dim=1,
        )

        # 5. 用 LLM 生成文本答案
        return self.language_model(
            inputs_embeds=multimodal_embeddings,
            labels=labels,
        )